# 06 - OU-TFG
TFG with the OU guess. Strengths tuned at t* = 100, then only the last **t\*** steps are guided. Run `02` and `04` first.

Needs `ou_guidance/` and `resnet_cifar10.pt` in `My Drive/tfg_ou/`. Runtime: T4 GPU.
If the install cell asks to restart, restart and run from the top.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tfg_ou'      # code, checkpoint and results

In [ ]:
# pinned: a newer diffusers clashes with transformers
!pip install -q diffusers==0.35.2 transformers==4.55.2 "huggingface-hub>=0.34,<1.0" torch-fidelity

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/ou_guidance')
import helpers
helpers.ROOT = ROOT
from helpers import run, search, show_results, diagnose

In [ ]:
LABELS = [8]        # CIFAR10 classes (8 = ship)
N = 100             # images per final run
N_SEARCH = 64       # images per search trial (as in the TFG repo)
SWEEP = [100, 60, 40, 30, 20, 10, 5]     # t*: guided last steps (100 = all)
SEARCH_PER_K = False                      # True: tune for every t* (much slower)

In [ ]:
for label in LABELS:
    best = search('ou', 'tfg', label=label, guided_steps=100, n=N_SEARCH)       # tune rho, mu, gamma at t* = 100
    print('best settings:', best)
    for k in SWEEP:
        settings = search('ou', 'tfg', label=label, guided_steps=k, n=N_SEARCH) if SEARCH_PER_K else best
        run('ou', label=label, n=N, guided_steps=k, **settings)

## Results

In [ ]:
show_results(kinds=['unconditional', 'TFG', 'mean-only', 'OU-TFG'])